# 01 · Data exploration: DVF géolocalisées, Paris (75)

**Goal:** understand the raw data before cleaning it. Every cleaning rule in Phase 3 should be justified by something we see here.

Questions answered in this notebook:
1. How many rows per year, and what columns do we have?
2. What kinds of transactions (`nature_mutation`) and properties (`type_local`) are in the data?
3. How often are surfaces or coordinates missing?
4. How often does one sale (`id_mutation`) span several rows (apartment + cellar + parking)?
5. What does a naive price per m² look like, and why can't we use it as is?

> Data: *Demandes de valeurs foncières géolocalisées* (DGFiP / Etalab), Licence Ouverte. Only aggregates are shown here; no address-level rows are published.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(RAW_DIR.glob("dvf_75_*.csv.gz"))
print(f"{len(files)} files found in {RAW_DIR}")
for f in files:
    print(f"  {f.name}  {f.stat().st_size / 1e6:.1f} MB")

## 1. Load all years

Codes such as `code_commune` or `code_postal` are read as text, otherwise pandas would drop leading zeros or turn them into floats.

In [ ]:
TEXT_COLUMNS = ["id_mutation", "code_commune", "code_postal", "code_departement", "id_parcelle", "code_type_local"]

frames = []
for f in files:
    year = int(f.stem.split("_")[-1].split(".")[0])
    df_year = pd.read_csv(f, dtype={c: "string" for c in TEXT_COLUMNS}, low_memory=False)
    df_year["source_year"] = year
    frames.append(df_year)

raw = pd.concat(frames, ignore_index=True)
raw["date_mutation"] = pd.to_datetime(raw["date_mutation"], errors="coerce")

print(f"Total rows: {len(raw):,}")
print(f"Columns ({raw.shape[1]}): {list(raw.columns)}")

## 2. Rows and sales per year

In [ ]:
per_year = raw.groupby("source_year").agg(
    rows=("id_mutation", "size"),
    sales=("id_mutation", "nunique"),
    first_date=("date_mutation", "min"),
    last_date=("date_mutation", "max"),
)
per_year["rows_per_sale"] = per_year["rows"] / per_year["sales"]
per_year

**Observation:** 420,058 rows for 207,365 distinct sales over 2021–2025, so about 2 rows per sale. Volume peaked in 2022 (47,240 sales), fell 23.5% to a low in 2024 (36,154, the year interest rates bit hardest), then recovered 12.9% in 2025 (40,811). Every year covers early January to late December, so all 5 years are complete.

## 3. Types of transaction: `nature_mutation`

Counted per **sale** (`id_mutation`), not per row, because one sale can have several rows.

In [ ]:
sales = raw.drop_duplicates("id_mutation")
nature = sales["nature_mutation"].value_counts(dropna=False).to_frame("sales")
nature["share_%"] = 100 * nature["sales"] / nature["sales"].sum()
nature

**Observation:** 98.6% of sales are ordinary `Vente`. Off-plan sales (VEFA, 0.6%), exchanges (0.56%), auctions (0.23%), building-land sales and expropriations are small and priced under different rules. Keeping `Vente` only costs about 1.4% of sales and makes prices comparable.

## 4. Types of property: `type_local`

In [ ]:
type_local = raw["type_local"].value_counts(dropna=False).to_frame("rows")
type_local["share_%"] = 100 * type_local["rows"] / type_local["rows"].sum()
type_local

Rows with an empty `type_local` are usually land parcels or lots without a building description.

**Observation:** Dependencies (cellars, parking spaces) are the most common row type (47.0%), slightly ahead of apartments (45.5%). Commercial premises are 6.4%, and houses are almost absent in Paris (0.22%). So almost half the rows are not homes and must not be counted as sales on their own.

## 5. Missing values in the key fields

In [ ]:
KEY_COLUMNS = [
    "date_mutation", "nature_mutation", "valeur_fonciere", "type_local",
    "surface_reelle_bati", "nombre_pieces_principales", "code_commune",
    "longitude", "latitude",
]
missing = raw[KEY_COLUMNS].isna().mean().mul(100).round(2).to_frame("missing_%")
missing

In [ ]:
# Same check, but only on apartment rows: these are the rows we will actually model.
apts = raw[raw["type_local"] == "Appartement"]
print(f"Apartment rows: {len(apts):,}")
apts[KEY_COLUMNS].isna().mean().mul(100).round(2).to_frame("missing_%_apartments")

In [ ]:
# Very small surfaces: likely storage rooms, data errors or 'chambres de bonne'.
bins = [0, 9, 15, 25, 40, 60, 100, 200, float("inf")]
surface_buckets = pd.cut(apts["surface_reelle_bati"], bins=bins, right=False)
surface_buckets.value_counts(sort=False, dropna=False).to_frame("apartment_rows")

**Observation:** `surface_reelle_bati` is missing on 48% of all rows, but only on 6 apartment rows: the gaps are dependencies, which have no built surface. Apartments are almost complete. 1.16% have no price, and 0.17% have no coordinates. 1,760 apartment rows (0.9%) are under 9 m², below the legal minimum for a home, so the `>= 9 m²` rule removes very little.

## 6. One sale, several rows

In DVF, one sale (`id_mutation`) is split into one row per lot: an apartment, its cellar (*Dépendance*), a parking space, etc.
**The price `valeur_fonciere` is repeated on every row.** Summing it per sale would multiply the real price.

In [ ]:
rows_per_sale = raw.groupby("id_mutation").size()
dist = rows_per_sale.clip(upper=10).value_counts().sort_index().to_frame("sales")
dist.index = [f"{i}" if i < 10 else "10+" for i in dist.index]
dist.index.name = "rows_in_sale"
dist["share_%"] = 100 * dist["sales"] / dist["sales"].sum()
dist

In [ ]:
# Is valeur_fonciere really the same on every row of a sale?
price_variants = raw.groupby("id_mutation")["valeur_fonciere"].nunique()
print(f"Sales with one single price across their rows: {(price_variants <= 1).mean():.2%}")
print(f"Sales with several different prices:          {(price_variants > 1).mean():.2%}")

In [ ]:
# What is inside each sale? Count apartments, dependencies, houses and commercial premises per id_mutation.
composition = (
    raw.assign(type_local=raw["type_local"].fillna("(none)"))
    .pivot_table(index="id_mutation", columns="type_local", values="source_year", aggfunc="size", fill_value=0)
)
print("Columns:", list(composition.columns))

n_apts = composition.get("Appartement", 0)
has_dep = composition.get("Dépendance", 0) > 0
other_types = composition.drop(columns=["Appartement", "Dépendance", "(none)"], errors="ignore").sum(axis=1) > 0

summary = pd.Series({
    "sales with no apartment": (n_apts == 0).sum(),
    "exactly 1 apartment, nothing else": ((n_apts == 1) & ~has_dep & ~other_types).sum(),
    "exactly 1 apartment + dependencies only": ((n_apts == 1) & has_dep & ~other_types).sum(),
    "exactly 1 apartment + house/commercial": ((n_apts == 1) & other_types).sum(),
    "2 or more apartments": (n_apts >= 2).sum(),
}, name="sales").to_frame()
summary["share_%"] = 100 * summary["sales"] / len(composition)
summary

**Observation:** Only 46% of sales are a single row. `valeur_fonciere` is identical on every row of 100% of sales, which confirms it is the total sale price repeated per lot: summing it would inflate prices. 72.7% of sales contain exactly one apartment (27.7% alone, plus 45.0% with cellar/parking). Allowing dependencies keeps 93,348 sales that an "apartment only" rule would wrongly drop. 4.1% (8,538) contain 2+ apartments: their price cannot be split by apartment, so they are excluded.

## 7. Arrondissement codes

Paris arrondissements use INSEE codes **75101 to 75120** in `code_commune`, while postal codes are 75001 to 75020 (plus 75116 for the 16th).
We will map `code_commune` to an arrondissement number 1 to 20.

In [ ]:
codes = raw.drop_duplicates("id_mutation")["code_commune"].value_counts().sort_index().to_frame("sales")
print(f"{len(codes)} distinct code_commune values")
codes.T

**Observation:** All 20 INSEE codes 75101–75120 are present. Volume varies a lot by arrondissement: from 2,285 sales in the 1st up to 21,275 in the 15th, the largest residential arrondissement. Thin samples in central arrondissements will make their medians noisier.

## 8. A naive price per m², and why we can't trust it yet

Restricted to the simplest case: `Vente` sales with exactly one apartment row and nothing else.
This is only a first look; the real cleaning happens in SQL in Phase 3.

In [ ]:
simple_ids = composition.index[(n_apts == 1) & ~has_dep & ~other_types]
simple = raw[
    raw["id_mutation"].isin(simple_ids)
    & (raw["type_local"] == "Appartement")
    & (raw["nature_mutation"] == "Vente")
].copy()
simple["price_per_m2"] = simple["valeur_fonciere"] / simple["surface_reelle_bati"]

print(f"Simple apartment sales: {len(simple):,}")
simple["price_per_m2"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
simple["price_per_m2"].clip(upper=30_000).plot.hist(bins=100, ax=ax, color="#3b6ea5")
ax.set_title("Naive price per m², simple apartment sales (clipped at 30,000 €/m²)")
ax.set_xlabel("€ per m²")
ax.set_ylabel("Sales")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_naive_price_per_m2_hist.png", dpi=150)
plt.show()

**Observation:** Even on the simplest sales, the naive €/m² is unusable as is. The median is a credible 10,182 €/m² (IQR 8,385–11,980), but the tails are absurd: minimum 0 €, 1st percentile 20 €, maximum 413,043 €/m². The histogram shows a spike of roughly 2,000 sales near 0 €/m², most likely symbolic or family transfers, or a price recorded for only part of a lot. The 5th percentile (2,117 €/m²) is still far below any real Paris price, so the 1st–99th percentile filter per arrondissement will be checked carefully in Phase 3.

## 9. Summary for Phase 3

A short table of the numbers that justify each cleaning rule. Copy this output into the chat.

In [ ]:
checks = pd.Series({
    "total rows": len(raw),
    "distinct sales (id_mutation)": raw["id_mutation"].nunique(),
    "share of sales that are 'Vente'": f"{(sales['nature_mutation'] == 'Vente').mean():.1%}",
    "apartment rows": len(apts),
    "apartment rows with surface < 9 m²": int((apts["surface_reelle_bati"] < 9).sum()),
    "apartment rows with missing surface": int(apts["surface_reelle_bati"].isna().sum()),
    "sales with 2+ apartments": int((n_apts >= 2).sum()),
    "sales with 1 apartment (+ dependencies allowed)": int(((n_apts == 1) & ~other_types).sum()),
    "naive €/m² 1st percentile": round(simple["price_per_m2"].quantile(0.01)),
    "naive €/m² median": round(simple["price_per_m2"].median()),
    "naive €/m² 99th percentile": round(simple["price_per_m2"].quantile(0.99)),
}, name="value")
checks.to_frame()

## Key takeaways

- **The data is complete and rich:** 207,365 sales over 5 full years, with surface and coordinates on almost every apartment row.
- **One sale ≠ one row.** The price is repeated on every lot of a sale (100% of cases), so all analysis must work per `id_mutation`, never on raw rows.
- **"One apartment per sale" is the right unit.** It keeps 72.7% of sales. Dependencies must be allowed, otherwise 45% of apartment sales would be lost. Multi-apartment sales (4.1%) are excluded because their price can't be split.
- **Outliers are real and large:** prices near 0 €/m² and above 400,000 €/m² exist. A per-arrondissement percentile filter is needed, and medians rather than means.
- **Market context:** volume fell 23.5% from 2022 to 2024, then recovered 12.9% in 2025. Phase 4 will show whether prices followed.